Adição de uma LLM para dar respostas mais claras da documentação do nosso projeto

In [1]:
!pip install -U transformers accelerate sentencepiece
!pip install sentence-transformers chromadb pypdf

In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import chromadb

In [3]:
pdf_path = "TCC CaoRadar - Oficial.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    content = page.extract_text()

    if content:
        text += content + "\n"

print("PDF carregado!")
print(len(text))

PDF carregado!
196832


In [4]:
chunk_size = 800
chunk_overlap = 150

chunks = []

for i in range(0, len(text), chunk_size - chunk_overlap):

    chunk = text[i:i + chunk_size]

    if len(chunk) > 100:
        chunks.append(chunk)

print("Quantidade de chunks:", len(chunks))

Quantidade de chunks: 303


In [5]:
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Modelo de embeddings carregado!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo de embeddings carregado!


In [6]:
client = chromadb.Client()

collection = client.create_collection(
    name="tcc_rag"
)

print("Banco vetorial criado!")

Banco vetorial criado!


In [7]:
embeddings = model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True
)

print("Embeddings criados!")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeddings criados!


In [8]:
for i, chunk in enumerate(chunks):

    collection.add(
        documents=[chunk],
        embeddings=[embeddings[i].tolist()],
        ids=[str(i)]
    )

print("Chunks armazenados!")

Chunks armazenados!


In [9]:
# CÉLULA 1 — Instala
!pip install groq -q

In [15]:
from groq import Groq

# Crie sua chave gratuita em: console.groq.com
client_groq = Groq(api_key="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")
print("Groq configurado!")

Groq configurado!


In [ ]:
# CÉLULA 3 — Loop de perguntas
while True:

    query = input("\nDigite sua pergunta (ou 'sair'): ")

    if query.lower() == "sair":
        break

    query_embedding = model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    context = "\n".join(results["documents"][0])

    response = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Você é um assistente especializado no TCC 'Cão Radar'. Responda sempre em português de forma clara e objetiva, baseado apenas no contexto fornecido."
            },
            {
                "role": "user",
                "content": f"Contexto:\n{context}\n\nPergunta: {query}"
            }
        ],
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()

    print("\n💬 RESPOSTA:")
    print(answer)
    print("\n" + "=" * 60)


Digite sua pergunta (ou 'sair'): Qual objetivo do tcc ?

💬 RESPOSTA:
O objetivo do TCC (Trabalho de Conclusão de Curso) "Cão Radar" é desenvolver uma ferramenta tecnológica que ajude na localização de cães perdidos ou roubados, utilizando Visão Computacional e Inteligência Artificial (IA) para pareamento e busca ativa.

Essa ferramenta visa inverter a lógica da busca manual e passiva para uma busca ativa e inteligente, utilizando imagens e dados de qualidade fornecidos pelos tutores dos cães. O objetivo é criar uma solução eficaz e eficiente para ajudar a reencontrar cães perdidos ou roubados, minimizando o sofrimento e a incerteza dos tutores e das familias.


Digite sua pergunta (ou 'sair'): Qual arquitetura do backend

💬 RESPOSTA:
A arquitetura do backend utilizada no projeto Cão Radar é baseada em duas tecnologias principais:

1. Java/Spring Boot: utilizada como tecnologia principal para desenvolver o back-end.
2. Python: utilizado como tecnologia para o microsserviço.

Juntamente